In [1]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../'))

from neuro_fuzzy_toolbox import ANFIS

# Testing

In [2]:
input = torch.tensor([[0.4398, 0.6070, 0.4809, 0.5871, 0.1615],
                      [0.7754, 0.5999, 0.7143, 0.6740, 0.3542],
                      [0.3313, 0.5182, 0.7479, 0.9098, 0.1208],
                      [0.8538, 0.3372, 0.2242, 0.9966, 0.5197],
                      [0.3438, 0.0063, 0.3992, 0.5114, 0.2140],
                      [0.3011, 0.6426, 0.6886, 0.6399, 0.7811],
                      [0.8961, 0.6595, 0.3033, 0.6119, 0.7812],
                      [0.1599, 0.5517, 0.8914, 0.4670, 0.6410],
                      [0.2305, 0.9696, 0.1346, 0.0170, 0.1085],
                      [0.8064, 0.8868, 0.4890, 0.9350, 0.5317]])



In [3]:
model = ANFIS(x_train=input,
                   init_fuzzy_rules=3,
                   outputs=2,
                   output_type='multiclass',
                   rule_reduced=True)

In [4]:
model.consequents_structure

[         c0 (x0)   c1 (x1)   c2 (x2)   c3 (x3)   c4 (x4)        c5
 rule 1  0.685389  0.428881 -0.152773 -0.208594 -0.538030 -0.507035
 rule 2  0.840157  0.419338 -0.866433  0.796501 -0.476472 -0.312556
 rule 3  0.047394  0.369836 -0.422792 -0.424595  0.214234  0.247389,
          c0 (x0)   c1 (x1)   c2 (x2)   c3 (x3)   c4 (x4)        c5
 rule 1  0.404942 -0.480044 -0.846877 -0.815024 -0.213107  0.590916
 rule 2  0.824505  0.106610 -0.989651  0.885471  0.799045  0.296697
 rule 3 -0.194665 -0.730695  0.091222 -0.025683 -0.120990 -0.012155]

In [5]:
output = model(input)
output

tensor([[0.3517, 0.6483],
        [0.3030, 0.6970],
        [0.3578, 0.6422],
        [0.2289, 0.7711],
        [0.3281, 0.6719],
        [0.2023, 0.7977],
        [0.1991, 0.8009],
        [0.2344, 0.7656],
        [0.4791, 0.5209],
        [0.7259, 0.2741]], grad_fn=<SoftmaxBackward0>)

In [6]:
model.predict(input)

array([[1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [0]])

# Gradient

## Rule reduced ANFIS

### Regression

#### Single output

In [7]:
x = torch.rand(10, 2)
y = torch.rand(10)
y

tensor([0.9695, 0.4452, 0.9033, 0.5989, 0.8058, 0.1677, 0.9581, 0.0818, 0.7165,
        0.6575])

In [8]:
model = ANFIS(x, init_fuzzy_rules=3, outputs=1, output_type='regression', rule_reduced=True)

In [9]:
model.get_premises().shape

torch.Size([2, 3, 3])

In [10]:
model.show_premises_structure()

                a (x0)  b (x0)    c (x0)    a (x1)  b (x1)    c (x1)
Fuzzy rule 1  0.179564     8.0  0.007670  0.173863     8.0  0.189787
Fuzzy rule 2  0.179564     8.0  0.366797  0.173863     8.0  0.537513
Fuzzy rule 3  0.179564     8.0  0.725925  0.173863     8.0  0.885238


In [11]:
model.get_consequents().shape

torch.Size([1, 3, 3])

In [12]:
model.show_consequents_structure()

- Output 1:
         c0 (x0)   c1 (x1)        c2
rule 1  0.648976  0.231666  0.793145
rule 2 -0.564208  0.460332 -0.447528
rule 3  0.479407 -0.729514 -0.834223




In [13]:
loader = data.DataLoader(data.TensorDataset(x, y), batch_size = 2)
x = loader.dataset.tensors[0]
y = loader.dataset.tensors[1]

In [14]:
model.predict(x)

array([[ 1.0010422 ],
       [-0.40313348],
       [-0.96122795],
       [-0.12390552],
       [-0.17055318],
       [-0.59904057],
       [-0.3402171 ],
       [-0.39470533],
       [-0.48833457],
       [ 0.8645873 ]], dtype=float32)

In [15]:
model(x)

tensor([[ 1.0010],
        [-0.4031],
        [-0.9612],
        [-0.1239],
        [-0.1706],
        [-0.5990],
        [-0.3402],
        [-0.3947],
        [-0.4883],
        [ 0.8646]], grad_fn=<TBackward0>)

In [16]:
y

tensor([0.9695, 0.4452, 0.9033, 0.5989, 0.8058, 0.1677, 0.9581, 0.0818, 0.7165,
        0.6575])

In [17]:
print(model._fuzzification_layer._premises.grad)

None


In [18]:
print(model._consequent_layer._consequents.grad)

None


In [19]:
model.predict(x)

array([[ 1.0010422 ],
       [-0.40313348],
       [-0.96122795],
       [-0.12390552],
       [-0.17055318],
       [-0.59904057],
       [-0.3402171 ],
       [-0.39470533],
       [-0.48833457],
       [ 0.8645873 ]], dtype=float32)

In [20]:
model(x)

tensor([[ 1.0010],
        [-0.4031],
        [-0.9612],
        [-0.1239],
        [-0.1706],
        [-0.5990],
        [-0.3402],
        [-0.3947],
        [-0.4883],
        [ 0.8646]], grad_fn=<TBackward0>)

In [21]:
model(x).squeeze()

tensor([ 1.0010, -0.4031, -0.9612, -0.1239, -0.1706, -0.5990, -0.3402, -0.3947,
        -0.4883,  0.8646], grad_fn=<SqueezeBackward0>)

In [22]:
loss = nn.functional.mse_loss(model(x).squeeze(), y)
loss

tensor(0.9668, grad_fn=<MseLossBackward0>)

In [23]:
loss.backward()

In [24]:
print(model._fuzzification_layer._premises.grad)

tensor([[[ 3.3440e-03, -2.6267e-05,  2.3526e-03],
         [-1.3368e+00,  1.2408e-02,  8.6621e-01],
         [ 1.3140e+00, -3.6537e-02, -3.8073e-01]],

        [[ 3.2279e-03,  2.9104e-03,  5.7294e-02],
         [-1.2051e+00,  1.8066e-02, -6.0129e-01],
         [ 2.3238e-02, -1.5010e-04, -1.7280e-02]]])


In [25]:
print(model._consequent_layer._consequents.grad)

tensor([[[ 0.0018,  0.0189,  0.0448],
         [-0.4242, -0.7746, -1.2447],
         [-0.2715, -0.2531, -0.3840]]])


#### Multiple outputs

In [27]:
x = torch.rand(10, 3)
y = torch.rand(10, 2)
y

tensor([[0.6215, 0.3403],
        [0.0571, 0.8661],
        [0.4204, 0.1916],
        [0.6693, 0.4810],
        [0.7640, 0.1440],
        [0.7096, 0.4045],
        [0.2869, 0.9066],
        [0.8656, 0.1906],
        [0.5675, 0.9757],
        [0.9372, 0.6925]])

In [28]:
model = ANFIS(x, init_fuzzy_rules=3, outputs=2, output_type='regression', rule_reduced=True)

In [29]:
model.get_premises().shape

torch.Size([3, 3, 3])

In [30]:
model.show_premises_structure()

                a (x0)  b (x0)    c (x0)    a (x1)  b (x1)    c (x1)  \
Fuzzy rule 1  0.209676     8.0  0.046557  0.206249     8.0  0.102428   
Fuzzy rule 2  0.209676     8.0  0.465909  0.206249     8.0  0.514926   
Fuzzy rule 3  0.209676     8.0  0.885261  0.206249     8.0  0.927425   

                a (x2)  b (x2)    c (x2)  
Fuzzy rule 1  0.241578     8.0  0.004841  
Fuzzy rule 2  0.241578     8.0  0.487997  
Fuzzy rule 3  0.241578     8.0  0.971152  


In [31]:
model.get_consequents().shape

torch.Size([2, 3, 4])

In [32]:
model.show_consequents_structure()

- Output 1:
         c0 (x0)   c1 (x1)   c2 (x2)        c3
rule 1  0.843020 -0.767695 -0.272160  0.593926
rule 2 -0.551586  0.229400 -0.394844 -0.189470
rule 3  0.181271  0.968545  0.378550  0.903756


- Output 2:
         c0 (x0)   c1 (x1)   c2 (x2)        c3
rule 1 -0.848042 -0.849382 -0.820115  0.791329
rule 2  0.378532 -0.465796  0.949350 -0.744515
rule 3  0.968574 -0.892563 -0.136015 -0.794726




## ANFIS

In [16]:
x = torch.rand(10, 2)
y = torch.randint(0, 4, (10,))

model = ANFIS(x, init_fuzzy_rules=3, outputs=4, output_type='multiclass')
loader = data.DataLoader(data.TensorDataset(x, y), batch_size = 2)

x = loader.dataset.tensors[0]
y = loader.dataset.tensors[1]

In [17]:
model.get_consequents().shape

torch.Size([4, 9, 3])

In [18]:
model.get_consequents()

tensor([[[ 0.4962,  0.1600,  0.3183],
         [-0.2938, -0.7863, -0.1584],
         [ 0.4256, -0.2650, -0.7060],
         [-0.2675,  0.3904, -0.3630],
         [-0.2153, -0.3411, -0.0142],
         [-0.9110,  0.3035, -0.0551],
         [-0.4285, -0.7588,  0.4580],
         [ 0.5665,  0.4035,  0.7892],
         [ 0.8850,  0.7406, -0.1451]],

        [[-0.8272,  0.7733,  0.2563],
         [ 0.6468, -0.0561, -0.2039],
         [ 0.1245,  0.3075,  0.9616],
         [-0.2878,  0.4385,  0.9876],
         [-0.6454, -0.9439,  0.1139],
         [-0.6118,  0.7056, -0.4067],
         [ 0.5247,  0.7073, -0.4591],
         [ 0.3527,  0.2154, -0.4986],
         [ 0.5252,  0.1457, -0.1073]],

        [[ 0.8757,  0.5523,  0.4385],
         [ 0.7439, -0.1549,  0.1313],
         [-0.1496,  0.4568,  0.6516],
         [-0.7456, -0.3824,  0.6333],
         [-0.7749,  0.1064, -0.5803],
         [-0.6767,  0.9029,  0.8084],
         [ 0.8834, -0.2299,  0.9101],
         [ 0.1922,  0.2602, -0.7026],
        

In [19]:
model.predict(x)

array([[0],
       [1],
       [0],
       [3],
       [0],
       [0],
       [2],
       [2],
       [2],
       [3]])

In [20]:
model(x)

tensor([[0.4904, 0.2176, 0.0955, 0.1965],
        [0.0508, 0.3893, 0.2919, 0.2680],
        [0.5982, 0.1239, 0.0900, 0.1879],
        [0.2007, 0.1167, 0.1029, 0.5796],
        [0.5922, 0.1247, 0.0909, 0.1921],
        [0.4759, 0.2178, 0.0998, 0.2066],
        [0.2937, 0.1771, 0.3835, 0.1458],
        [0.1538, 0.2901, 0.4073, 0.1488],
        [0.2583, 0.1857, 0.4001, 0.1560],
        [0.0876, 0.1142, 0.3052, 0.4930]], grad_fn=<SoftmaxBackward0>)

In [21]:
y

tensor([2, 0, 1, 1, 3, 3, 3, 2, 0, 2])

In [22]:
print(model._fuzzification_layer._premises.grad)

None


In [23]:
loss = nn.functional.cross_entropy(model(x), y)
loss.backward()
print(model._fuzzification_layer._premises.grad)

tensor([[[-9.4481e-02,  1.7884e-04, -8.5942e-02],
         [ 5.4955e-02, -6.9494e-06, -1.4781e-01],
         [-7.9064e-06,  4.1703e-06, -1.9372e-04]],

        [[-3.4668e-03, -1.1649e-05, -3.9692e-03],
         [ 2.0937e-01, -7.6233e-04,  9.7726e-02],
         [-1.3353e-02, -7.1866e-05,  1.6152e-02]]])


In [25]:
model._consequent_layer._consequents.grad.shape

torch.Size([4, 9, 3])

In [26]:
model._consequent_layer._consequents.grad

tensor([[[-6.3205e-03, -3.3730e-03, -1.1942e-02],
         [ 1.8376e-03,  1.7652e-03,  4.2167e-03],
         [-2.1582e-03, -4.4249e-03, -4.8492e-03],
         [-7.1915e-04, -3.2321e-04, -1.4918e-03],
         [ 1.1464e-03,  1.0074e-03,  1.5636e-03],
         [ 8.4285e-04,  1.1316e-03,  1.5904e-03],
         [ 1.9592e-08,  1.2467e-08,  2.2501e-08],
         [ 2.1269e-02,  1.3516e-02,  2.4479e-02],
         [ 1.7319e-02,  1.5335e-02,  1.8482e-02]],

        [[ 2.4871e-03,  8.6122e-04,  5.7279e-03],
         [ 5.3933e-03,  4.3434e-03,  1.2022e-02],
         [ 1.4133e-03,  2.7296e-03,  3.0914e-03],
         [ 1.6032e-04,  7.2063e-05,  3.3259e-04],
         [-7.7494e-03, -6.6188e-03, -1.0767e-02],
         [ 1.0508e-03,  1.4774e-03,  2.0424e-03],
         [-6.5283e-09, -4.1577e-09, -7.3979e-09],
         [-9.1324e-03, -5.8120e-03, -1.0385e-02],
         [ 4.8258e-03,  4.2933e-03,  5.1819e-03]],

        [[ 6.9587e-03,  2.3102e-03,  1.6214e-02],
         [-9.9381e-03, -8.3353e-03, -2.2225e-0